 K-Fold Cross Validation for Multiple Linear Regression (Least Square Error Fit) 
Download the dataset regarding USA House Price Prediction from the following link: 
https://drive.google.com/file/d/1O_NwpJT-8xGfU_-3llUl2sgPu0xllOrX/view?usp=sharing 
Load the dataset and Implement 5- fold cross validation for multiple linear regression (using 
least square error fit). 
Steps: 
1. Divide the dataset into input features (all columns except price) and output variable 
(price) 
2. Scale the values of input features. 
3. Divide input and output features into five folds. 
4. Run five iterations, in each iteration consider one-fold as test set and remaining four sets 
as training set. Find the beta (𝛽) matrix, predicted values, and R2_score for each iteration 
using least square error fit. 
5. Use the best value of (𝛽) matrix (for which R2_score is maximum), to train the regressor 
for 70% of data and test the performance for remaining 30% data. 

In [22]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_csv("./Data/USA_Housing.csv")
df.head()

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price
0,79545.45857,5.682861,7.009188,4.09,23086.80050,1.059034e+06
1,79248.64245,6.002900,6.730821,3.09,40173.07217,1.505891e+06
2,61287.06718,5.865890,8.512727,5.13,36882.15940,1.058988e+06
3,63345.24005,7.188236,5.586729,3.26,34310.24283,1.260617e+06
4,59982.19723,5.040555,7.839388,4.23,26354.10947,6.309435e+05


In [24]:
df.columns

Index(['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms',
       'Avg. Area Number of Bedrooms', 'Area Population', 'Price'],
      dtype='object')

In [25]:
columns = df.columns
for column in columns[:-1]:
    min = df[column].min()
    max = df[column].max()
    den = max-min
    df[column] = (df[column]-min)/den
df.head()

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price
0,0.686822,0.441986,0.501502,0.464444,0.329942,1.059034e+06
1,0.683521,0.488538,0.464501,0.242222,0.575968,1.505891e+06
2,0.483737,0.468609,0.701350,0.695556,0.528582,1.058988e+06
3,0.506630,0.660956,0.312430,0.280000,0.491549,1.260617e+06
4,0.469223,0.348556,0.611851,0.495556,0.376988,6.309435e+05


In [26]:
x = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [27]:
x

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population
0,0.686822,0.441986,0.501502,0.464444,0.329942
1,0.683521,0.488538,0.464501,0.242222,0.575968
2,0.483737,0.468609,0.701350,0.695556,0.528582
3,0.506630,0.660956,0.312430,0.280000,0.491549
4,0.469223,0.348556,0.611851,0.495556,0.376988
...,...,...,...,...,...
4995,0.475738,0.754359,0.385619,0.324444,0.326351
4996,0.675097,0.633450,0.444024,0.448889,0.366362
4997,0.507135,0.670026,0.208534,0.028889,0.476515
4998,0.558419,0.420389,0.517579,0.764444,0.611282


In [28]:
y

0       1.059034e+06
1       1.505891e+06
2       1.058988e+06
3       1.260617e+06
4       6.309435e+05
            ...     
4995    1.060194e+06
4996    1.482618e+06
4997    1.030730e+06
4998    1.198657e+06
4999    1.298950e+06
Name: Price, Length: 5000, dtype: float64

In [29]:
foldsize = int(len(x)/5)
len(x),foldsize

(5000, 1000)

In [36]:
folds = []
outputs = []
for i in range(0,len(x),foldsize):
    print(i,i+foldsize)
    temp1 = x.iloc[i:i+foldsize:]
    temp2 = y.iloc[i:i+foldsize:]
    folds.append(temp1)
    outputs.append(temp2)
folds[3],outputs[3]

0 1000
1000 2000
2000 3000
3000 4000
4000 5000


(      Avg. Area Income  Avg. Area House Age  Avg. Area Number of Rooms  \
 3000          0.601991             0.766240                   0.512604   
 3001          0.641973             0.285625                   0.644760   
 3002          0.540153             0.567609                   0.638675   
 3003          0.565501             0.552963                   0.364734   
 3004          0.759553             0.509040                   0.470516   
 ...                ...                  ...                        ...   
 3995          0.365545             0.458034                   0.218307   
 3996          0.526623             0.751607                   0.460756   
 3997          0.398695             0.271328                   0.597146   
 3998          0.421777             0.348402                   0.316320   
 3999          0.561379             0.474629                   0.473697   
 
       Avg. Area Number of Bedrooms  Area Population  
 3000                      0.728889        

In [37]:
params = []
r2s = []
valr2s = []
for i in range(0,5):
    train_set = np.array(pd.concat([folds[j] for j in range(0,len(folds)) if j != i]))
    y_set_train = np.array(pd.concat([outputs[j] for j in range(0,len(outputs)) if j != i]))
    val_Set = np.array(folds[i])
    y_val_Set = np.array(outputs[i])
    train_set = np.c_[np.ones(train_set.shape[0]), train_set]
    val_Set = np.c_[np.ones(val_Set.shape[0]), val_Set]
    ###training
    betas = np.matmul(np.linalg.inv(np.matmul(train_set.transpose(),train_set)),np.matmul(train_set.transpose(),y_set_train))
    train_predictions = np.matmul(train_set,betas)
    r2 = 1 - (np.sum((train_predictions-y_set_train)**2))/((np.sum((y_set_train-np.mean(y_set_train))**2)))
    params.append(betas)
    r2s.append(r2)
    print("Train R2 Score: "+str(r2))

    ### validating
    val_predictions = np.matmul(val_Set,betas)
    r2 = 1 - (np.sum((val_predictions-y_val_Set)**2))/((np.sum((y_val_Set-np.mean(y_val_Set))**2)))
    valr2s.append(r2)
    print("Validation R2 Score: "+str(r2))

Train R2 Score: 0.918041930988685
Validation R2 Score: 0.9175899480765111
Train R2 Score: 0.9173221048597836
Validation R2 Score: 0.9203015496401126
Train R2 Score: 0.9186414845108315
Validation R2 Score: 0.9152429915320013
Train R2 Score: 0.917230839559029
Validation R2 Score: 0.920850383697766
Train R2 Score: 0.9189737185134472
Validation R2 Score: 0.9138111758717504


In [39]:
valr2s

[0.9175899480765111,
 0.9203015496401126,
 0.9152429915320013,
 0.920850383697766,
 0.9138111758717504]

In [40]:
best_index = np.argmax(np.array(valr2s))
best_index

3

In [41]:
best_params = params[best_index]
best_params

array([-1414578.41177661,  1939689.35012524,  1140921.38723207,
         895697.76567999,     4099.75517049,  1058866.32351347])

In [46]:
####on full data
d70 = int(0.7*len(x))
len(x),d70

(5000, 3500)

In [70]:
x_tr = x.iloc[0:d70,:]
y_tr = y.iloc[0:d70]

In [71]:
x_t = x.iloc[d70:len(x),:]
y_t = y.iloc[d70:len(y)]
len(x_t),len(y_t)

(1500, 1500)

In [72]:
foldsize = int(len(x_tr)/5)
foldsize

700

In [73]:
folds = []
outputs = []
for i in range(0,len(x_tr),foldsize):
    print(i,i+foldsize)
    temp1 = x_tr.iloc[i:i+foldsize:]
    temp2 = y_tr.iloc[i:i+foldsize:]
    folds.append(temp1)
    outputs.append(temp2)
folds[3],outputs[3],len(folds[3]),len(outputs[3])

0 700
700 1400
1400 2100
2100 2800
2800 3500


(      Avg. Area Income  Avg. Area House Age  Avg. Area Number of Rooms  \
 2100          0.672974             0.448024                   0.750916   
 2101          0.571248             0.261063                   0.459040   
 2102          0.648584             0.611885                   0.471158   
 2103          0.418426             0.435648                   0.185650   
 2104          0.713795             0.400932                   0.497530   
 ...                ...                  ...                        ...   
 2795          0.850848             0.487144                   0.537336   
 2796          0.502103             0.578837                   0.588262   
 2797          0.604283             0.514583                   0.626983   
 2798          0.429360             0.458083                   0.516326   
 2799          0.365601             0.689592                   0.509306   
 
       Avg. Area Number of Bedrooms  Area Population  
 2100                      0.911111        

In [74]:
params = []
r2s = []
valr2s = []
for i in range(0,5):
    train_set = np.array(pd.concat([folds[j] for j in range(0,len(folds)) if j != i]))
    y_set_train = np.array(pd.concat([outputs[j] for j in range(0,len(outputs)) if j != i]))
    val_Set = np.array(folds[i])
    y_val_Set = np.array(outputs[i])
    train_set = np.c_[np.ones(train_set.shape[0]), train_set]
    val_Set = np.c_[np.ones(val_Set.shape[0]), val_Set]
    ###training
    betas = np.matmul(np.linalg.inv(np.matmul(train_set.transpose(),train_set)),np.matmul(train_set.transpose(),y_set_train))
    train_predictions = np.matmul(train_set,betas)
    r2 = 1 - (np.sum((train_predictions-y_set_train)**2))/((np.sum((y_set_train-np.mean(y_set_train))**2)))
    params.append(betas)
    r2s.append(r2)
    print("Train R2 Score: "+str(r2))

    ### validating
    val_predictions = np.matmul(val_Set,betas)
    r2 = 1 - (np.sum((val_predictions-y_val_Set)**2))/((np.sum((y_val_Set-np.mean(y_val_Set))**2)))
    valr2s.append(r2)
    print("Validation R2 Score: "+str(r2))

Train R2 Score: 0.9182809466814498
Validation R2 Score: 0.9170056919950382
Train R2 Score: 0.9178595105083732
Validation R2 Score: 0.9185110303568388
Train R2 Score: 0.9166093090021042
Validation R2 Score: 0.9226206475535721
Train R2 Score: 0.9190618676346699
Validation R2 Score: 0.9135866311216853
Train R2 Score: 0.9186561188719435
Validation R2 Score: 0.9151040123364312


In [75]:
best_index = np.argmax(np.array(valr2s))
best_index

2

In [76]:
best_betas = params[best_index]
best_betas

array([-1.42153572e+06,  1.94337224e+06,  1.14317264e+06,  9.25642260e+05,
        7.57877046e+02,  1.04537110e+06])

In [77]:
y_t = np.array(y_t)

In [78]:
x_t_1 = np.c_[np.ones(x_t.shape[0]), x_t]
x_t_1.shape

(1500, 6)

In [79]:
test_predictions = np.matmul(x_t_1,best_betas)
test_predictions.shape

(1500,)

In [80]:
r2 = 1 - (np.sum((test_predictions-y_t)**2))/((np.sum((y_t-np.mean(y_t))**2)))
r2

0.9175880102439207

Concept of Validation set for Multiple Linear Regression (Gradient Descent 
Optimization) 
Consider the same dataset of Q1, rather than dividing the dataset into five folds, divide the 
dataset into training set (56%), validation set (14%), and test set (30%). 
Consider four different values of learning rate i.e. {0.001,0.01,0.1,1}. Compute the values of 
regression coefficients for each value of learning rate after 1000 iterations.  
For each set of regression coefficients, compute R2_score for validation and test set and find the 
best value of regression coefficients.

In [85]:
indices = np.arange(0,len(x),dtype=int)
indices

array([   0,    1,    2, ..., 4997, 4998, 4999])

In [93]:
np.random.shuffle(indices)
indices

array([1804,  230, 2087, ..., 4844, 2179, 2990])

In [92]:
x_array = np.array(x)
y_array = np.array(y)
x_array.shape, y_array.shape

((5000, 5), (5000,))

In [94]:
x_array = x_array[indices,:]
y_array = y_array[indices]
x_array.shape, y_array.shape

((5000, 5), (5000,))

In [95]:
train_limit = int(0.56*len(x_array))
validation_limit = int(0.14*len(x_array))
test_limit = int(0.30*len(x_array))
train_limit,validation_limit,test_limit

(2800, 700, 1500)

In [96]:
x_tr = x_array[0:train_limit,:]
y_tr = y_array[0:train_limit]
x_v = x_array[train_limit:train_limit+validation_limit,:]
y_v = y_array[train_limit:train_limit+validation_limit]
x_t = x_array[train_limit+validation_limit:train_limit+validation_limit+test_limit,:]
y_t = y_array[train_limit+validation_limit:train_limit+validation_limit+test_limit]

In [97]:
epochs = 1000

In [98]:
x_t.shape[1]+1

6

In [99]:
x_tr.shape[0]

2800

Training R2: -12.339549070262802 Validation R2: -12.173249629238658


In [ ]:
r2vals = []
alphas = [0.001,0.01,0.1,1]
params = []
for alpha in alphas:
    betas = np.zeros(x_tr.shape[1]+1)
    x_tr1 = np.c_[np.ones(x_tr.shape[0]),x_tr]    
    n = len(x_tr1)
    for i in range(0,epochs):

        ##training
        predictions = np.matmul(x_tr1,betas)
        errors = predictions - y_tr
        gradient = 1/n*np.matmul(x_tr1.transpose(),errors)
        betas = betas - alpha*gradient
        r2t = 1 - (np.sum((predictions-y_tr)**2))/((np.sum((y_tr-np.mean(y_tr))**2)))
        print("Epoch: " + str(i) + " Training R2: "+str(r2t))

    ##validation
    x_v1 = np.c_[np.ones(x_v.shape[0]),x_v]    
    val_predictions = np.matmul(x_v1,betas)
    r2v = 1 - (np.sum((val_predictions-y_v)**2))/((np.sum((y_v-np.mean(y_v))**2)))
    print("Validation R2: "+str(r2v))
    r2vals.append(r2v)
    params.append(betas)

Epoch: 0 Training R2: -12.339549070262802
Epoch: 1 Training R2: -12.281577767527851
Epoch: 2 Training R2: -12.223871003208032
Epoch: 3 Training R2: -12.166427569898868
Epoch: 4 Training R2: -12.109246265706718
Epoch: 5 Training R2: -12.052325894223628
Epoch: 6 Training R2: -11.995665264502296
Epoch: 7 Training R2: -11.939263191031147
Epoch: 8 Training R2: -11.88311849370952
Epoch: 9 Training R2: -11.82722999782297
Epoch: 10 Training R2: -11.771596534018697
Epoch: 11 Training R2: -11.716216938281056
Epoch: 12 Training R2: -11.661090051907214
Epoch: 13 Training R2: -11.606214721482887
Epoch: 14 Training R2: -11.551589798858213
Epoch: 15 Training R2: -11.497214141123717
Epoch: 16 Training R2: -11.443086610586398
Epoch: 17 Training R2: -11.389206074745916
Epoch: 18 Training R2: -11.335571406270892
Epoch: 19 Training R2: -11.282181482975318
Epoch: 20 Training R2: -11.229035187795073
Epoch: 21 Training R2: -11.176131408764542
Epoch: 22 Training R2: -11.123469038993347
Epoch: 23 Training R2: 

In [102]:
r2vals

[0.21921929920632,
 0.4523696728504252,
 0.8021069465843109,
 -5.0616271487614915e+218]

In [103]:
alphas

[0.001, 0.01, 0.1, 1]

In [104]:
epochs = 5000
r2vals = []
alphas = [0.001,0.01,0.1,1]
params = []
for alpha in alphas:
    betas = np.zeros(x_tr.shape[1]+1)
    x_tr1 = np.c_[np.ones(x_tr.shape[0]),x_tr]    
    n = len(x_tr1)
    for i in range(0,epochs):

        ##training
        predictions = np.matmul(x_tr1,betas)
        errors = predictions - y_tr
        gradient = 1/n*np.matmul(x_tr1.transpose(),errors)
        betas = betas - alpha*gradient
        r2t = 1 - (np.sum((predictions-y_tr)**2))/((np.sum((y_tr-np.mean(y_tr))**2)))
        print("Epoch: " + str(i) + " Training R2: "+str(r2t))

    ##validation
    x_v1 = np.c_[np.ones(x_v.shape[0]),x_v]    
    val_predictions = np.matmul(x_v1,betas)
    r2v = 1 - (np.sum((val_predictions-y_v)**2))/((np.sum((y_v-np.mean(y_v))**2)))
    print("Validation R2: "+str(r2v))
    r2vals.append(r2v)
    params.append(betas)

Epoch: 0 Training R2: -12.339549070262802
Epoch: 1 Training R2: -12.281577767527851
Epoch: 2 Training R2: -12.223871003208032
Epoch: 3 Training R2: -12.166427569898868
Epoch: 4 Training R2: -12.109246265706718
Epoch: 5 Training R2: -12.052325894223628
Epoch: 6 Training R2: -11.995665264502296
Epoch: 7 Training R2: -11.939263191031147
Epoch: 8 Training R2: -11.88311849370952
Epoch: 9 Training R2: -11.82722999782297
Epoch: 10 Training R2: -11.771596534018697
Epoch: 11 Training R2: -11.716216938281056
Epoch: 12 Training R2: -11.661090051907214
Epoch: 13 Training R2: -11.606214721482887
Epoch: 14 Training R2: -11.551589798858213
Epoch: 15 Training R2: -11.497214141123717
Epoch: 16 Training R2: -11.443086610586398
Epoch: 17 Training R2: -11.389206074745916
Epoch: 18 Training R2: -11.335571406270892
Epoch: 19 Training R2: -11.282181482975318
Epoch: 20 Training R2: -11.229035187795073
Epoch: 21 Training R2: -11.176131408764542
Epoch: 22 Training R2: -11.123469038993347
Epoch: 23 Training R2: 

c:\Users\Gaurish Garg\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_21288\3653339755.py:16: RuntimeWarning: overflow encountered in square
  r2t = 1 - (np.sum((predictions-y_tr)**2))/((np.sum((y_tr-np.mean(y_tr))**2)))


Epoch: 1579 Training R2: -inf
Epoch: 1580 Training R2: -inf
Epoch: 1581 Training R2: -inf
Epoch: 1582 Training R2: -inf
Epoch: 1583 Training R2: -inf
Epoch: 1584 Training R2: -inf
Epoch: 1585 Training R2: -inf
Epoch: 1586 Training R2: -inf
Epoch: 1587 Training R2: -inf
Epoch: 1588 Training R2: -inf
Epoch: 1589 Training R2: -inf
Epoch: 1590 Training R2: -inf
Epoch: 1591 Training R2: -inf
Epoch: 1592 Training R2: -inf
Epoch: 1593 Training R2: -inf
Epoch: 1594 Training R2: -inf
Epoch: 1595 Training R2: -inf
Epoch: 1596 Training R2: -inf
Epoch: 1597 Training R2: -inf
Epoch: 1598 Training R2: -inf
Epoch: 1599 Training R2: -inf
Epoch: 1600 Training R2: -inf
Epoch: 1601 Training R2: -inf
Epoch: 1602 Training R2: -inf
Epoch: 1603 Training R2: -inf
Epoch: 1604 Training R2: -inf
Epoch: 1605 Training R2: -inf
Epoch: 1606 Training R2: -inf
Epoch: 1607 Training R2: -inf
Epoch: 1608 Training R2: -inf
Epoch: 1609 Training R2: -inf
Epoch: 1610 Training R2: -inf
Epoch: 1611 Training R2: -inf
Epoch: 161

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_21288\3653339755.py:14: RuntimeWarning: overflow encountered in matmul
  gradient = 1/n*np.matmul(x_tr1.transpose(),errors)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_21288\3653339755.py:15: RuntimeWarning: invalid value encountered in subtract
  betas = betas - alpha*gradient


Epoch: 3627 Training R2: nan
Epoch: 3628 Training R2: nan
Epoch: 3629 Training R2: nan
Epoch: 3630 Training R2: nan
Epoch: 3631 Training R2: nan
Epoch: 3632 Training R2: nan
Epoch: 3633 Training R2: nan
Epoch: 3634 Training R2: nan
Epoch: 3635 Training R2: nan
Epoch: 3636 Training R2: nan
Epoch: 3637 Training R2: nan
Epoch: 3638 Training R2: nan
Epoch: 3639 Training R2: nan
Epoch: 3640 Training R2: nan
Epoch: 3641 Training R2: nan
Epoch: 3642 Training R2: nan
Epoch: 3643 Training R2: nan
Epoch: 3644 Training R2: nan
Epoch: 3645 Training R2: nan
Epoch: 3646 Training R2: nan
Epoch: 3647 Training R2: nan
Epoch: 3648 Training R2: nan
Epoch: 3649 Training R2: nan
Epoch: 3650 Training R2: nan
Epoch: 3651 Training R2: nan
Epoch: 3652 Training R2: nan
Epoch: 3653 Training R2: nan
Epoch: 3654 Training R2: nan
Epoch: 3655 Training R2: nan
Epoch: 3656 Training R2: nan
Epoch: 3657 Training R2: nan
Epoch: 3658 Training R2: nan
Epoch: 3659 Training R2: nan
Epoch: 3660 Training R2: nan
Epoch: 3661 Tr

In [106]:
r2vals

[0.40857862401138223, 0.6713439246278441, 0.9116817082993264, nan]

In [107]:
epochs = 10000
r2vals = []
alphas = [0.001,0.01,0.1,1]
params = []
for alpha in alphas:
    betas = np.zeros(x_tr.shape[1]+1)
    x_tr1 = np.c_[np.ones(x_tr.shape[0]),x_tr]    
    n = len(x_tr1)
    for i in range(0,epochs):

        ##training
        predictions = np.matmul(x_tr1,betas)
        errors = predictions - y_tr
        gradient = 1/n*np.matmul(x_tr1.transpose(),errors)
        betas = betas - alpha*gradient
        r2t = 1 - (np.sum((predictions-y_tr)**2))/((np.sum((y_tr-np.mean(y_tr))**2)))
        print("Epoch: " + str(i) + " Training R2: "+str(r2t))

    ##validation
    x_v1 = np.c_[np.ones(x_v.shape[0]),x_v]    
    val_predictions = np.matmul(x_v1,betas)
    r2v = 1 - (np.sum((val_predictions-y_v)**2))/((np.sum((y_v-np.mean(y_v))**2)))
    print("Validation R2: "+str(r2v))
    r2vals.append(r2v)
    params.append(betas)

Epoch: 0 Training R2: -12.339549070262802
Epoch: 1 Training R2: -12.281577767527851
Epoch: 2 Training R2: -12.223871003208032
Epoch: 3 Training R2: -12.166427569898868
Epoch: 4 Training R2: -12.109246265706718
Epoch: 5 Training R2: -12.052325894223628
Epoch: 6 Training R2: -11.995665264502296
Epoch: 7 Training R2: -11.939263191031147
Epoch: 8 Training R2: -11.88311849370952
Epoch: 9 Training R2: -11.82722999782297
Epoch: 10 Training R2: -11.771596534018697
Epoch: 11 Training R2: -11.716216938281056
Epoch: 12 Training R2: -11.661090051907214
Epoch: 13 Training R2: -11.606214721482887
Epoch: 14 Training R2: -11.551589798858213
Epoch: 15 Training R2: -11.497214141123717
Epoch: 16 Training R2: -11.443086610586398
Epoch: 17 Training R2: -11.389206074745916
Epoch: 18 Training R2: -11.335571406270892
Epoch: 19 Training R2: -11.282181482975318
Epoch: 20 Training R2: -11.229035187795073
Epoch: 21 Training R2: -11.176131408764542
Epoch: 22 Training R2: -11.123469038993347
Epoch: 23 Training R2: 

c:\Users\Gaurish Garg\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_21288\3078470972.py:16: RuntimeWarning: overflow encountered in square
  r2t = 1 - (np.sum((predictions-y_tr)**2))/((np.sum((y_tr-np.mean(y_tr))**2)))


Epoch: 2364 Training R2: -inf
Epoch: 2365 Training R2: -inf
Epoch: 2366 Training R2: -inf
Epoch: 2367 Training R2: -inf
Epoch: 2368 Training R2: -inf
Epoch: 2369 Training R2: -inf
Epoch: 2370 Training R2: -inf
Epoch: 2371 Training R2: -inf
Epoch: 2372 Training R2: -inf
Epoch: 2373 Training R2: -inf
Epoch: 2374 Training R2: -inf
Epoch: 2375 Training R2: -inf
Epoch: 2376 Training R2: -inf
Epoch: 2377 Training R2: -inf
Epoch: 2378 Training R2: -inf
Epoch: 2379 Training R2: -inf
Epoch: 2380 Training R2: -inf
Epoch: 2381 Training R2: -inf
Epoch: 2382 Training R2: -inf
Epoch: 2383 Training R2: -inf
Epoch: 2384 Training R2: -inf
Epoch: 2385 Training R2: -inf
Epoch: 2386 Training R2: -inf
Epoch: 2387 Training R2: -inf
Epoch: 2388 Training R2: -inf
Epoch: 2389 Training R2: -inf
Epoch: 2390 Training R2: -inf
Epoch: 2391 Training R2: -inf
Epoch: 2392 Training R2: -inf
Epoch: 2393 Training R2: -inf
Epoch: 2394 Training R2: -inf
Epoch: 2395 Training R2: -inf
Epoch: 2396 Training R2: -inf
Epoch: 239

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_21288\3078470972.py:14: RuntimeWarning: overflow encountered in matmul
  gradient = 1/n*np.matmul(x_tr1.transpose(),errors)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_21288\3078470972.py:15: RuntimeWarning: invalid value encountered in subtract
  betas = betas - alpha*gradient


Epoch: 3241 Training R2: nan
Epoch: 3242 Training R2: nan
Epoch: 3243 Training R2: nan
Epoch: 3244 Training R2: nan
Epoch: 3245 Training R2: nan
Epoch: 3246 Training R2: nan
Epoch: 3247 Training R2: nan
Epoch: 3248 Training R2: nan
Epoch: 3249 Training R2: nan
Epoch: 3250 Training R2: nan
Epoch: 3251 Training R2: nan
Epoch: 3252 Training R2: nan
Epoch: 3253 Training R2: nan
Epoch: 3254 Training R2: nan
Epoch: 3255 Training R2: nan
Epoch: 3256 Training R2: nan
Epoch: 3257 Training R2: nan
Epoch: 3258 Training R2: nan
Epoch: 3259 Training R2: nan
Epoch: 3260 Training R2: nan
Epoch: 3261 Training R2: nan
Epoch: 3262 Training R2: nan
Epoch: 3263 Training R2: nan
Epoch: 3264 Training R2: nan
Epoch: 3265 Training R2: nan
Epoch: 3266 Training R2: nan
Epoch: 3267 Training R2: nan
Epoch: 3268 Training R2: nan
Epoch: 3269 Training R2: nan
Epoch: 3270 Training R2: nan
Epoch: 3271 Training R2: nan
Epoch: 3272 Training R2: nan
Epoch: 3273 Training R2: nan
Epoch: 3274 Training R2: nan
Epoch: 3275 Tr

In [108]:
r2vals

[0.4523644750284309, 0.8020457599189692, 0.9118602430458679, nan]

In [110]:
index = np.nanargmax(r2vals)
index

2

In [111]:
best_alpha = alphas[index]
best_alpha

0.1

In [112]:
best_params = params[index]
best_params

array([-1420757.33632194,  1935924.5078461 ,  1124960.2605968 ,
         919071.97821889,     6662.8027308 ,  1064408.51927216])

In [113]:
x_t1 = np.c_[np.ones(x_t.shape[0]),x_t]    
test_predictions = np.matmul(x_t1,best_params)
r2v = 1 - (np.sum((test_predictions-y_t)**2))/((np.sum((y_t-np.mean(y_t))**2)))
print("Test R2: "+str(r2v))


Test R2: 0.9205123253830161


In [114]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()
model.fit(x_tr, y_tr)

y_pred = model.predict(x_t)

r2 = r2_score(y_t, y_pred)

print("Sklearn Test R2:", r2)

Sklearn Test R2: 0.9205114448054168
